# EXP5 - MVE Analysis

In [ ]:
# Cell 1: imports and config
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import pymannkendall as mk
from pathlib import Path

from src.config import (
    PROCESSED_DATA_DIR,
    OUTPUT_DATA_DIR,
    OUTPUT_STATIC_ATTR,
    DATA_START_YEAR,
    DATA_END_YEAR,
    TRAIN_END_YEAR,
)

# Analysis parameters
WINDOW = 15                       
WINDOWS_ROBUST = [10, 20]
N_MEMBERS = 10
N_BOOTSTRAP = 1000
RNG = np.random.default_rng(42)

# Input paths
TOPOGRAPH_LSTM = OUTPUT_DATA_DIR / 'topographic_lstm'
OBS_PATH = PROCESSED_DATA_DIR / 'combined_streamflow.csv'
STATIC_PATH = OUTPUT_STATIC_ATTR

# Output paths
MVE_RESULTS_DIR = OUTPUT_DATA_DIR / 'results_MVE'
FIG_DIR = MVE_RESULTS_DIR / 'figures'

# FIG_DIR.mkdir(parents=True, exist_ok=True)

## Math Foundation: decomposition and notation

**Setup**

- $y_t$ — observed flow at time $t$
- $f_m(x_t),\ m = 1,\dots,M,\ M = 10$ — frozen ensemble member predictions (point predictors)
- $\hat\mu_t = \frac{1}{M}\sum_{m=1}^{M} f_m(x_t)$ — ensemble mean
- $r_t = y_t - \hat\mu_t$ — residual
- $\hat b(x) = \mathbb{E}[\,r \mid x\,]$ — conditional bias (binned/smoothed mean residual, Part 1)

**Law of total variance** (the quantity being decomposed)

$$
\underbrace{\mathrm{Var}(y \mid x)}_{\text{total}}
= \underbrace{\mathbb{E}_{\theta}\!\big[\mathrm{Var}(y \mid x, \theta)\big]}_{\text{aleatoric},\ \sigma_a^2(x)}
+ \underbrace{\mathrm{Var}_{\theta}\!\big[\mathbb{E}(y \mid x, \theta)\big]}_{\text{epistemic},\ \sigma_e^2(x)}
$$

**Empirical estimators (point-predictor ensemble)**

Epistemic — between-member variance, available pointwise:

$$
\hat\sigma_e^2(x_t) = \frac{1}{M-1}\sum_{m=1}^{M}\big(f_m(x_t) - \hat\mu_t\big)^2
$$

Aleatoric — conditional residual variance, the head's target:

$$
\hat\sigma_a^2(x) = \mathrm{Var}\big[\,y - \hat\mu \mid x\,\big]
= \underbrace{\mathbb{E}[\,r^2 \mid x\,]}_{\mathrm{MSR}(x)} - \hat b(x)^2
$$

estimated by binning/smoothing over similar $x$ (flow percentile or regime).

Total predictive variance:

$$
\hat\sigma_{\text{tot}}^2(x) = \hat\sigma_a^2(x) + \hat\sigma_e^2(x)
$$

Variance head (next stage):

$$
\hat\sigma_{\text{head}}^2(x_t) \approx \sigma_a^2(x_t)
$$

a smooth pointwise function fit to the empirical $\hat\sigma_a^2$.

**Notes**

1. **Aleatoric is the residual _variance_, not the MSR.** Since $\mathrm{MSR}(x) = \sigma_a^2(x) + b(x)^2$, raw MSR double-counts squared bias — hence the Part 1 gate must clear first. Where $\hat b(x) \approx 0$, $\hat\sigma_a^2(x) \approx \widehat{\mathrm{MSR}}(x)$.
2. **Epistemic is added, never subtracted.** Total $= \sigma_a^2 + \sigma_e^2$. Aleatoric comes from the mean's residual variance; epistemic is layered on top for the total.
3. **Pointwise vs conditional.** $\hat\sigma_e^2$ exists at every $t$ (all $M$ members are present); $\hat\sigma_a^2$ needs many residuals, so it is binned/windowed, not per-timestep. This is why epistemic anchors the AR / heat-dome events pointwise while aleatoric-at-events waits for the head.
4. **Point predictors, so the head is necessary.** Members emit no per-member variance, so the LTV aleatoric term $\mathbb{E}_\theta[\mathrm{Var}(y\mid x,\theta)]$ is empty: the ensemble gives $\hat\sigma_e^2$ directly but cannot give $\sigma_a^2$, which is recovered from residuals here and modelled by the head next.
5. **Conventions.** Use $M-1$ (ten members is a small sample); define $y, \hat\mu, r$ in the Part 1 transform space, and transform back explicitly if reporting native-flow variance, since the split is not invariant to the log/NSE\* mapping.

## Part 1 - Mean/residuals analysis from Frozen Mean head

Notes about this analysis:

We want to compare the ensemble mean ($\hat\mu$) against the observed flow $y$. We then are hoping to extract $\hat\sigma$ from a variance head post-hoc. 

## Bias:
### Residuals vs Fitted plot: With a mean estimate line and spread band
Check heteroscedasticity of predictions.

### Binned bias and spread by flow percentile: mean residual with confidence interval per bin, and residual standard deviation per bin
Shows whether bias concentrates in the high- and low-flow tails and how spread scales with flow.

### Predicted vs observed scatter, linear and log panels, with the 1:1 line 
The cloud falling below 1:1 at the top end is peak under-prediction; the log panel exposes low-flow behavior the linear panel hides.

## Residual Structures: 

### Residual distributions — histogram with a Normal overlay plus a Normal QQ-plot, split by flow regime (high-flow residuals separately from low-flow)
Symmetric and Gaussian means a plain MVE head will dom skew or heavy tails, especially if regime-dependent. Otherwise tread carefully.

### Residual ACF - with a residual time-series strip for a representative window. 
Leftover autocorrelation means the mean missed some dynamics and that residuals aren't iid.

## OOD Anchoring and Bridge:

### Event hydrographs for the AR and heat-dome windows - Obs vs $\hat\mu$ with ensemble envelope, residual plotted beneath. 
The mean's baseline behavior and bias at the two extremes, which everything in the variance/OOD stage reference. 

## Part 2 - Variance Foundations

Notes about this analysis:

We split the spread around $\hat\mu$ into epistemic (between-member, from the 10 frozen members) and aleatoric (the remainder $\hat\sigma$ will model), identify the structure that remainder takes, and locate it where it matters: tails and the two events. 

Everything is conditioned on the Part 1 bias so $\text{bias}^2$ doesn't leak into the aleatoric term, and carries bootstrap CIs since the remainder is a noisy difference of variances.



## Decomposition: epistemic vs aleatoric

### Between-member variance (epistemic): per-timestep spread of the 10 members
Var across the frozen members at each x — the model uncertainty read straight from the ensemble, no head needed.

### Total vs epistemic: smoothed $(y - \hat\mu)^2$ against between-member variance
Total residual variance = epistemic + aleatoric remainder; the remainder is the head's target and budget. Subtract the Part 1 conditional bias first (smooth with WINDOW, WINDOWS_ROBUST for sensitivity).

### Variance partition by flow percentile: stacked epistemic/aleatoric shares per bin, bootstrap CIs
Mirrors the Part 1 binned-bias figure — which component dominates per regime, and where the remainder is too small to read.

## something else tbd